# Ordinary Least Squares Regression Coefficients (Linear Algebra)

This notebook derives Ordinary Least Squares (OLS) regression coefficients 
manually using the matrix left inverse formula.
Most practitioners use OLS through black-box libraries. This notebook strips that away and derives the coefficients directly from first principles, specifically the 
closed-form solution:

**β = (XᵀX)⁻¹Xᵀy**

**X** is the design matrix and **y** is the target vector. The last part also includes the same coefficients calculated using the standard statistical library `sklearn`.
The derivation of this left inverse formula is outside the scope of this notebook.

## The data
The design matrix is constructed from 5 years of daily price return data, 
structured as an autoregressive model with 5 lags (AR(5)). Each row 
represents a point in time, and each column represents a lagged return — 
meaning we are using the past 5 returns to predict the current return.

This is the same mathematical structure underlying:
- Time series forecasting models
- Factor regression in quantitative finance
- The linear layer in neural networks

In [1]:
# Ordinary Least Squares Regression Coefficients (Linear Algebra)

# This notebook derives Ordinary Least Squares (OLS) regression coefficients 
# manually using the matrix left inverse formula.
# Most practitioners use OLS through black-box libraries. This notebook strips that away and derives the coefficients directly from first principles, specifically the 
# closed-form solution:

# **β = (XᵀX)⁻¹Xᵀy**

# **X** is the design matrix and **y** is the target vector. The last part also includes the same coefficients calculated using the standard statistical library `sklearn`.
# The derivation of this left inverse formula is outside the scope of this notebook.

# ## The data
# The design matrix is constructed from 5 years of daily price return data, 
# structured as an autoregressive model with 5 lags (AR(5)). Each row 
# represents a point in time, and each column represents a lagged return — 
# meaning we are using the past 5 returns to predict the current return.

# This is the same mathematical structure underlying:
# - Time series forecasting models
# - Factor regression in quantitative finance
# - The linear layer in neural networks

# import libraries
import yfinance as yf
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

In [58]:
# download historical data for SP500 from Yahoo Finance
ticker = '^GSPC'
period = '5y'
data = yf.download(ticker, period = period, multi_level_index = False)

# drop unnecessary columns and calculate returns
data['Returns'] = data['Close'].pct_change()
data = data[['Close', 'Returns']].dropna()

# create autoregressive features (lag = 5 design matrix)
lags = 5
X = pd.DataFrame(index = data.index)

X['intercept'] = 1 # add intercept term 
for lag in range(1, lags + 1):
    X[f'Lag_{lag}'] = data['Returns'].shift(lag)

X = X.dropna()

# target variable
y = data['Returns'].loc[X.index]

[*********************100%***********************]  1 of 1 completed


In [72]:
# compute OLS estimates using matrix algebra
beta = np.linalg.inv(X.T @ X) @ X.T @ y
beta = np.array(beta) # convert to numpy array for easier comparison

# coefficients using sklearn for comparison
model = LinearRegression(fit_intercept=False)
model.fit(X, y)
sklearn_beta = model.coef_
sklearn_beta

# print coefficients and check for errors
print(f'OLS inverse intercept = {np.round(beta[0], 4)}')

for i in range(1, len(beta)-1):
    print(f'OLS inverse beta_{i} = {np.round(beta[i], 4)}')

print('-------')
print(f'Sklearn intercept = {np.round(sklearn_beta[0], 4)}')

for i in range(1, len(sklearn_beta)-1):
    print(f'Sklearn beta_{i} = {np.round(sklearn_beta[i], 4)}')

print('-------')
print("Difference between OLS and sklearn coefficients:")
print(np.round(sklearn_beta - beta, 2 ))

OLS inverse intercept = 0.0005
OLS inverse beta_1 = -0.0247
OLS inverse beta_2 = -0.0061
OLS inverse beta_3 = -0.0699
OLS inverse beta_4 = -0.0417
-------
Sklearn intercept = 0.0005
Sklearn beta_1 = -0.0247
Sklearn beta_2 = -0.0061
Sklearn beta_3 = -0.0699
Sklearn beta_4 = -0.0417
-------
Difference between OLS and sklearn coefficients:
[ 0.  0. -0.  0.  0.  0.]
